In [ ]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/refs/heads/master/data/tinyshakespeare/input.txt

--2026-07-11 21:04:38--  https://raw.githubusercontent.com/karpathy/char-rnn/refs/heads/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.009s  

2026-07-11 21:04:38 (123 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [ ]:
dataset = open('input.txt', 'r', encoding='utf-8')
text = dataset.read()

In [ ]:
print(len(text))

1115394


In [ ]:
print(text[:500])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars)) #the first element is actually '\n' and not ' ' like andrej said
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [ ]:
encode_map = {ch:i for i,ch in enumerate(chars)} #maps characters to integers
decode_map = {i:ch for i,ch in enumerate(chars)} #maps integers back to characters

encode = lambda s: [encode_map[c] for c in s] #assigns each character of the string 's' to an int from the map 'encode_map'
decode = lambda l: ''.join(decode_map[c] for c in l) #gets the corresponding character for every integer in the list created by the 'encode_map' for the string

In [ ]:
print(encode('hello world'))
print(decode(encode('hello world'))) # first converting string to list of integers and then mapping back to characters joining them together

[46, 43, 50, 50, 53, 1, 61, 53, 56, 50, 42]
hello world


In [ ]:
import torch
data = torch.tensor(encode(text), dtype=torch.long) # coverting all of the input.txt into encoded list of integers for each character. torch.long makes sure it is stored as integers.
print(data.shape,data.dtype)
print(data[:500])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [ ]:
n = int(0.9* len(data))
train_data = data[:n]
val_data = data[n:]

In [ ]:
torch.manual_seed(1337) # to get same numbers as andrej (helps in croschecking)

batch_size = 4
block_size = 8

def get_batch(split_type):
  data = train_data if split_type == 'train' else val_data
  ix = torch.randint(len(data)-block_size, (batch_size,))
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])
  return x,y

xb, yb = get_batch('train')
print(xb.shape, '\n', xb)
print(yb.shape, '\n', yb)

torch.Size([4, 8]) 
 tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
torch.Size([4, 8]) 
 tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])


In [ ]:
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337) # to keep track with Andrej

class BigramLM(nn.Module):

  def __init__(self,vocab_size):
    super().__init__() # essentially to inherit the __init__() of the superclass (nn.Module)
    self.token_embedding_table = nn.Embedding(vocab_size,vocab_size)

  def forward(self, i, targets=None):
    logits = self.token_embedding_table(i) # (Batch, Time, Channel)
    if targets == None:
      loss = None
    else:
      B,T,C= logits.shape
      logits = logits.view(B*T,C)#3D to 2D compresses (4,8,65) to (32,65)
      targets = targets.view(B*T) #2D to 1D compresses (4,8) to (32,)
      loss = F.cross_entropy(logits,targets)
    return logits , loss

  def generate(self,i,max_new_tokens):
    for _ in range(max_new_tokens):
      logits, loss = self(i) #runs forward() on each i in the (4,8) xb
      logits = logits[:,-1,:] #takes only the last character of each of the 4 rows so shape is now (4,65)
      probs = F.softmax(logits,dim=-1) #converts logits to probabilities
      i_next = torch.multinomial(probs, num_samples = 1) #randomly picks one number (from the 65 available) weighted by the probabilities
      #i_next = torch.argmax(probs, dim=-1, keepdim=True)  # instead of torch.multinomial(probs, num_samples=1)
      i = torch.cat((i,i_next), dim=1) # concatenates the 4 new i_next (shape is (4,1)) to the existing xb (shape of xb goes from 4,8 to 4,9 and so on every iteration)
    return i

model = BigramLM(vocab_size)
logits, loss = model(xb,yb)
print(loss)

tensor(4.8786, grad_fn=<NllLossBackward0>)


In [ ]:
idx = torch.zeros((1,1), dtype = torch.long)
print(decode(model.generate(idx,100)[0].tolist()))


SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(),lr=0.001)

In [ ]:
batch_size = 32
for steps in range(10000):
  xb,yb = get_batch('train')
  logits,loss = model(xb,yb)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()

print(loss.item())

2.382369041442871


In [ ]:
print(decode(model.generate(i = torch.zeros((1,1), dtype = torch.long),max_new_tokens=500)[0].tolist()))


lso br. ave aviasurf my, yxMPZI ivee iuedrd whar ksth y h bora s be hese, woweee; the! KI 'de, ulseecherd d o blllando;LUCEO, oraingofof win!
RIfans picspeserer hee tha,
TOFonk? me ain ckntoty ded. bo'llll st ta d:
ELIS me hurf lal y, ma dus pe athouo
BEY:! Indy; by s afreanoo adicererupa anse tecorro llaus a!
OLeneerithesinthengove fal amas trr
TI ar I t, mes, n IUSt my w, fredeeyove
THek' merer, dd
We ntem lud engitheso; cer ize helorowaginte the?
Thak orblyoruldvicee chot, p,
Bealivolde Th li


In [ ]:
B,T,C = 4,8,32
x = torch.randn(B,T,C)

##Version 1

In [ ]:
# xbow = torch.zeros((B,T,C)) # bow = bag of words, since we are averaging all of the context prior to the current element.
# for b in range(B):
#   for t in range(T):
#     xprev = x[b,:t+1] #drops the b'th dimention so shape is t,c for each xprev. takes all pairs (since c is 2) upto t'th term for the current batch.
#     xbow[b,t] = torch.mean(xprev,0) # compresses the xprev along 0th position to form a (2,) shape that replaces the zeros at b,t in xbow.

##Version 2

In [ ]:
# w = torch.tril(torch.ones(T,T)) # tril creates a lower triangular matrix filled with ones in the diagonal and below. creates a 8x8 lower triangular matrix
# w = w / w.sum(1, keepdim=True) # dividing each 1 by summ of the total ones in that row. essentially dividing the whole row into equal elements summing to one instead of the ones.
# xbow2 = w @ x # matrix multiplication faster multiplying w (shaped T,T) and x (shaped B,T,C). since they are un equal, pytorch creates a B dimension so essentially replicates the T,T matrix B times to be able to multiply to x. This does (B,T,T) @ (B,T,C) = (B,T,C) matrix

##Version 3

In [ ]:
# tril = torch.tril(torch.ones(T,T))# lower triangular matrix of ones
# w = torch.zeros(T,T) #init w matrix with all  (this is not zeros in practice)
# w = w.masked_fill(tril == 0,float('-inf')) # fills matrix w with float('-inf') wherever the mask tril==0 is true (so upper triangle excluding diagonal) think of it like the future tokens cannot communicate with the past
# w = F.softmax(w,dim=-1) #softmax converts raw numbers into probabilities by row
# out = w @ x

# out.shape

torch.Size([4, 8, 32])

# Self written Version 4 (self-attention) with a couple hints

In [ ]:
import numpy as np

head_size = 16

key = nn.Linear(C,head_size, bias = False) # key nn taking 32 inputs outputting 16 number keys. (4,8,32) to (4,8,16)
query = nn.Linear(C,head_size, bias = False) # query nn taking 32 inputs outputting 16 number queries. (4,8,32) to (4,8,16)
value = nn.Linear(C,head_size, bias = False) # value nn taking 32 inputs outputting 16 number values. (4,8,32) to (4,8,16)
'''
we take no biases since bias's point is to make sure regardless of input, the neuron outputs some baseline value.
q,k and v are all just used for comparison between scores for tokens so adding an equal amount to each wont change how they are related but will add an extra parameter to consider
'''
q = query(x) # query asks "what this current token is looking for"
k = key(x) # key asks "what imformation is this current token providing"
v = value(x) # value at last asks "what information should this current token relay when being attended to"

w = q @ k.transpose(-1,-2) # transpose(-1,-2) swaps last 2 dimensions so (4,8,16) becomes (4,16,8) which is what we matrix multiply with (4,8,16) to get (4,8,8) which we want (query of each token interacting with every other token's key in the sequence)
w = w / np.sqrt(head_size)
'''
sqrt specifically since we need to scale down using std dev. imagine 16 coin tosses +1 for heads -1 for tails.
realistically a score of +/- 16 is extremely rare (think of normal distribution curve)
usually +4 or -6 somehting like that. similarly instead of divinding by variance we divide by std dev
which is sqrt of variance. variance here: since we are adding 16 terms together, variance increases 16x
std dev increases 4x so dividing by std dev gives terms roughly scaled to original terms.
'''
tril = torch.tril(torch.ones(T,T))# lower triangular matrix of ones
w = w.masked_fill(tril == 0,float('-inf')) # fills matrix w with float('-inf') wherever the mask tril==0 is true (so upper triangle excluding diagonal) think of it like the future tokens cannot communicate with the past
w = F.softmax(w,dim=-1) #softmax converts raw numbers into probabilities by row
out = w @ v
out.shape

torch.Size([4, 8, 16])

In [ ]:
class LayerNorm:

  def __init__(self, dim, eps = 1e-5):
    self.eps = eps
    # parameters (trained with backprop)
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)

  def __call__(self, x):
    xmean = x.mean(1, keepdim=True) # layer mean
    xvar = x.var(1, keepdim=True) # layer var
    xhat = (x-xmean)/torch.sqrt(xvar+self.eps)
    self.out = self.gamma * xhat + self.beta
    return self.out

  def parameters(self):
    return [self.gamma, self.beta]

In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F

batch_size = 64
block_size = 256
max_iters = 5000
eval_interval = 300
lr = 3e-4
eval_iters = 200
n_embed = 384
n_heads = 6
n_layers = 6
dropout = 0.2

torch.manual_seed(1337)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

dataset = open('input.txt', 'r', encoding='utf-8')
text = dataset.read()

chars = sorted(list(set(text)))  #the first element is actually '\n' and not ' ' like andrej said
vocab_size = len(chars)

encode_map = {ch:i for i,ch in enumerate(chars)} #maps characters to integers
decode_map = {i:ch for i,ch in enumerate(chars)} #maps integers back to characters
encode = lambda s: [encode_map[c] for c in s] #assigns each character of the string 's' to an int from the map 'encode_map'
decode = lambda l: ''.join(decode_map[c] for c in l) #gets the corresponding character for every integer in the list created by the 'encode_map' for the string

data = torch.tensor(encode(text), dtype=torch.long) # coverting all of the input.txt into encoded list of integers for each character. torch.long makes sure it is stored as integers.
n = int(0.9* len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split_type):
  data = train_data if split_type == 'train' else val_data
  ix = torch.randint(len(data)-block_size, (batch_size,))
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])
  x, y = x.to(device), y.to(device)
  return x,y

@torch.no_grad()
def estimate_loss():
  out = {}
  model.eval()
  for split in ['train','val']:
    losses = torch.zeros(eval_iters)
    for k in range(eval_iters):
      x,y = get_batch(split)
      logits,loss = model(x,y)
      losses[k] = loss.item()
    out[split] = losses.mean()
  model.train()
  return out

class Head(nn.Module):

  def __init__(self,head_size):
    super().__init__()
    self.key = nn.Linear(n_embed,head_size, bias = False) # key nn taking 32 inputs outputting 16 number keys. (4,8,32) to (4,8,16)
    self.query = nn.Linear(n_embed,head_size, bias = False) # query nn taking 32 inputs outputting 16 number queries. (4,8,32) to (4,8,16)
    self.value = nn.Linear(n_embed,head_size, bias = False) # value nn taking 32 inputs outputting 16 number values. (4,8,32) to (4,8,16)
    self.dropout = nn.Dropout(dropout)
    self.register_buffer('tril',torch.tril(torch.ones(block_size,block_size))) # lower triangular matrix of ones.
    '''
    register_buffer is PyTorch's way of saying: "this tensor belongs to the model and should travel with it (device, save/load)
    but it's not a learnable parameter, don't compute gradients for it, don't let the optimizer touch it."
    '''

  def forward(self,x):
    B,T,C = x.shape
    q = self.query(x) # query asks "what this current token is looking for"
    k = self.key(x) # key asks "what imformation is this current token providing"
    v = self.value(x) # value at last asks "what information should this current token relay when being attended to"

    w = q @ k.transpose(-1,-2) * C**-0.5 # transpose(-1,-2) swaps last 2 dimensions so (4,8,16) becomes (4,16,8) which is what we matrix multiply with (4,8,16) to get (4,8,8) which we want (query of each token interacting with every other token's key in the sequence)
    '''
    sqrt specifically since we need to scale down using std dev. imagine 16 coin tosses +1 for heads -1 for tails.
    realistically a score of +/- 16 is extremely rare (think of normal distribution curve)
    usually +4 or -6 somehting like that. similarly instead of divinding by variance we divide by std dev
    which is sqrt of variance. variance here: since we are adding 16 terms together, variance increases 16x
    std dev increases 4x so dividing by std dev gives terms roughly scaled to original terms.
    '''
    w = w.masked_fill(self.tril[:T,:T] == 0,float('-inf')) # fills matrix w with float('-inf') wherever the mask tril==0 is true (so upper triangle excluding diagonal) think of it like the future tokens cannot communicate with the past
    w = F.softmax(w,dim=-1) #softmax converts raw numbers into probabilities by row
    w = self.dropout(w)
    out = w @ v
    return out

class MultiHeadAttention(nn.Module):

  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)]) # in place of a regular list. essentially tells pytorch that this is a model part and parameters should be tracked. similar to register_buffer but for lists.
    self.proj = nn.Linear(n_embed,n_embed)
    self.dropout = nn.Dropout(dropout)

  def forward(self,x):
    out = torch.cat([h(x) for h in self.heads], dim = -1) #concatenates all the outputs of heads along the last dimension
    out = self.dropout(self.proj(out))
    return out

class FeedForward(nn.Module):

  def __init__(self, n_embed):
    super().__init__()
    self.net = nn.Sequential(
      nn.Linear(n_embed, 4*n_embed),
      nn.ReLU(),
      nn.Linear(4*n_embed,n_embed), # projection layer
      nn.Dropout(dropout),
    )

  def forward(self,x):
    return self.net(x)

class Block(nn.Module):

  def __init__(self, n_embed, num_heads):
    super().__init__()
    head_size = n_embed//num_heads # divides 32 by number of heads so when they are concatenated they become back to 32 regardless of number of heads
    self.sa = MultiHeadAttention(num_heads, head_size)
    self.ffwd = FeedForward(n_embed)
    self.ln1 = nn.LayerNorm(n_embed)
    self.ln2 = nn.LayerNorm(n_embed)

  def forward(self, x):
    x = x + self.sa(self.ln1(x)) # added a residual learning layer that passes gradient directly to input (add function passes gradients directly) since the model is too deep and suffers from vanishing gradients.
    x = x + self.ffwd(self.ln2(x))
    return x

class BigramLM(nn.Module):

  def __init__(self):
    super().__init__() # essentially to inherit the __init__() of the superclass (nn.Module)
    self.token_embedding_table = nn.Embedding(vocab_size,n_embed) # embedding table with 65 rows each with 32 values random numbers instead of 65
    self.positional_embedding_table = nn.Embedding(block_size,n_embed)
    self.blocks = nn.Sequential(*[Block(n_embed,n_heads) for _ in range(n_layers)])
    self.ln_f = nn.LayerNorm(n_embed)
    self.lm_head = nn.Linear(n_embed,vocab_size) # neural network layer with 32 inputs per layer and 65 such neurons leading to 65 outputs like before. no activation.

  def forward(self, i, targets=None):
    B,T = i.shape
    tok_emb = self.token_embedding_table(i) # (Batch, Time, Channel) takes a row of 32 numbers for each i
    pos_emb = self.positional_embedding_table(torch.arange(T,device=device)) #(T,C)
    x = tok_emb + pos_emb #(B,T,C)
    x = self.blocks(x)
    x = self.ln_f(x)
    logits = self.lm_head(x) # (B,T,vocab_size) feeds the 32 numbers as input to a linear NN layer with 65 neurons. each neuron has 32 weights, 1 for each input and 1 bias. total 65 outputs 1 per neuron.

    if targets == None:
      loss = None
    else:
      B,T,C= logits.shape
      logits = logits.view(B*T,C)#3D to 2D compresses (4,8,65) to (32,65)
      targets = targets.view(B*T) #2D to 1D compresses (4,8) to (32,)
      loss = F.cross_entropy(logits,targets)
    return logits , loss

  def generate(self,i,max_new_tokens):
    for _ in range(max_new_tokens):
      i_cond = i[:,-block_size:]
      logits, loss = self(i_cond) #runs forward() on each i in the (4,8) xb
      logits = logits[:,-1,:] #takes only the last character of each of the 4 rows so shape is now (4,65)
      probs = F.softmax(logits,dim=-1) #converts logits to probabilities
      i_next = torch.multinomial(probs, num_samples = 1) #randomly picks one number (from the 65 available) weighted by the probabilities
      #i_next = torch.argmax(probs, dim=-1, keepdim=True)  # instead of torch.multinomial(probs, num_samples=1)
      i = torch.cat((i,i_next), dim=1) # concatenates the 4 new i_next (shape is (4,1)) to the existing xb (shape of xb goes from 4,8 to 4,9 and so on every iteration)
    return i

model = BigramLM()
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(),lr=lr)

for iter in range(max_iters):
  if iter%eval_interval==0:
    losses = estimate_loss()
    print(f'step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}')
  xb,yb = get_batch('train')
  logits,loss = model(xb,yb)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()

context = torch.zeros((1,1),dtype = torch.long, device=device)
print(decode(model.generate(context,max_new_tokens=500)[0].tolist()))

Using device: cuda
step 0: train loss 4.2849, val loss 4.2823
step 300: train loss 2.3184, val loss 2.3467
step 600: train loss 1.8885, val loss 2.0022
step 900: train loss 1.6508, val loss 1.8092
step 1200: train loss 1.5197, val loss 1.7085
step 1500: train loss 1.4391, val loss 1.6394
step 1800: train loss 1.3756, val loss 1.5855
step 2100: train loss 1.3275, val loss 1.5587
step 2400: train loss 1.2917, val loss 1.5373
step 2700: train loss 1.2558, val loss 1.5143
step 3000: train loss 1.2269, val loss 1.5049
step 3300: train loss 1.2014, val loss 1.4929
step 3600: train loss 1.1810, val loss 1.4929
step 3900: train loss 1.1547, val loss 1.4874
step 4200: train loss 1.1320, val loss 1.4833
step 4500: train loss 1.1064, val loss 1.4779
step 4800: train loss 1.0870, val loss 1.4898

But with prison: I will steal fund to him.
Why lamentabroad is of that writ, say alore,
'Tis self my own matter, let no more honour.'

First Servingman:
Well, go, tell thy land: make her masters: I hear
h